# Mario Bros con aprendizaje por refuerzo: DQN vs PPO
*Esqueleto del notebook final (`model_development.ipynb`). Todas las cifras se generan desde los CSV crudos de `results/`; no se escriben a mano.*

## 1. El problema
*(P3, día 1)* Playtesting automático: un agente que juega niveles para ayudar a diseñadores a detectar dificultad, bloqueos y zonas problemáticas, y que sirve como simulación barata de jugadores.
- ¿Qué se quiere medir? Progreso por nivel, tasa de completado, dónde muere el agente.
- ¿Por qué RL? *(completar)*

## 2. Entorno y preprocesamiento (env-v1)
SIMPLE_MOVEMENT · frame skip 4 · escala de grises · 84×84 · stack de 4 · recompensa del entorno sin escalar.
Niveles: entrenamiento 1-1; no vistos 2-1 y 3-1 (opcional 1-2). Longitud de cada nivel en `levels.json`.

## 3. Algoritmos
- **DQN** (foco): replay buffer, red objetivo, epsilon-greedy. *(P2: justificar por qué encaja con acciones discretas y riesgos)*
- **PPO** (baseline): política estocástica, GAE, clipping. *(P2)*
- Hiperparámetros: ver `dqn/README.md` y `ppo/`.

## 4. Protocolo de evaluación
Mismo para todos los modelos: 1 episodio por semilla; validación 100–104 (1-1, solo para elegir checkpoint); evaluación final 200–209;
no-ops aleatorios 0–30; DQN epsilon-greedy fijo 0.05 (en `evaluate.py`), PPO muestreando la política; resultado secundario determinista.
Métrica principal: progreso = max_x / level_length; secundaria: tasa de completado.
**Limitación:** epsilon-greedy y muestreo de política no son aleatoriedades equivalentes.

In [ ]:
import pandas as pd, glob
from pathlib import Path

RESULTS = Path("results")
def load(pattern):
    files = sorted(glob.glob(str(RESULTS / pattern), recursive=True))
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True) if files else pd.DataFrame()

df = load("**/*.csv")
df.head()

## 5. Resultados
*(P2, día 6)* Tabla comparativa y curvas generadas desde `df`. Reportar nivel visto/no visto por separado.

In [ ]:
# Ejemplo de tabla comparativa (completar cuando haya resultados finales)
if not df.empty:
    summary = (df.groupby(["algo", "level", "prediction_mode"])
                 .agg(episodios=("episode", "count"),
                      max_x_medio=("max_x", "mean"),
                      completado=("completed", "mean"))
                 .round(3))
    display(summary)

## 6. Generalización, limitaciones y reajuste
*(P3)* Niveles no vistos solo se evalúan una vez al final. Si ningún agente completa 1-1, reportar progreso observado sin afirmar que completó el juego.
Reajuste: monitoreo del progreso en validación, umbral de alerta, reentrenamiento, validación frente a la versión anterior y rollback.